# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
# Fallback: if find_dotenv() didn't locate the .env (notebook launched from a different CWD),
# load it explicitly from this lab's folder.
if not OPENAI_API_KEY:
    from pathlib import Path
    lab_env = Path(r"C:\Users\sisaz\PycharmProjects\IronHackProjects\Labs\w5\d4\Labs\lab-sql-query-from-table-names\.env")
    load_dotenv(lab_env, override=True)
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

assert OPENAI_API_KEY, "OPENAI_API_KEY still not set"

In [3]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [4]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['employees', 'salary', 'studies', 'departments', 'projects'],
        'definition': [
            'Employee information including id, name, year of birth, sex, and department id.',
            'Salary records per employee per year, including the gross amount paid.',
            'Educational background of employees: institution, qualification, level, employee id, and year completed.',
            'Department catalog with department id, department name, and the building where it is located.',
            'Projects assigned to employees: project id, name, start date, end date, employee id, and status.'
        ]}
df = pd.DataFrame(data)
print(df)

         table                                         definition
0    employees  Employee information including id, name, year ...
1       salary  Salary records per employee per year, includin...
2      studies  Educational background of employees: instituti...
3  departments  Department catalog with department id, departm...
4     projects  Projects assigned to employees: project id, na...


In [5]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [6]:
print(text_tables)

employees: Employee information including id, name, year of birth, sex, and department id.
salary: Salary records per employee per year, including the gross amount paid.
studies: Educational background of employees: institution, qualification, level, employee id, and year completed.
departments: Department catalog with department id, department name, and the building where it is located.
projects: Projects assigned to employees: project id, name, start date, end date, employee id, and status.


In [7]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [8]:
# Test 1: a simple question that should only need ONE or TWO tables.
# Expectation: the model picks `employees` + `salary`.
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="Return the name of the best paid employee"
)

In [9]:
print(return_OAI(pqt1))

{
    "tables": ["employees", "salary"]
}


In [10]:
# Test 2: a harder question that needs THREE tables joined together.
# Expectation: the model picks `employees` + `salary` + `studies` (+ maybe `departments`).
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Return the average salary per department for employees who completed a Master's degree"
)

In [11]:
print(return_OAI(pqt3))

{
    "tables": ["employees", "salary", "studies", "departments"]
}


## More prompt variations

I want to push the model a bit and see when it picks the right tables and when it gets confused.
I'll try four kinds of questions:

1. **Easy** — answerable from one or two tables.
2. **Multi-table** — needs joins across several tables.
3. **Vague / subjective** — no clear table maps to it. Does the model invent something?
4. **Out-of-scope** — the data isn't there at all. Does the model admit it or hallucinate?

In [12]:
# A list of questions covering the four cases above.
# We loop through them, send each one to the model, and print the answer.
questions = [
    # 1) Easy — should only need `employees`.
    "Who is the youngest employee?",

    # 2) Multi-table — needs employees + departments + studies + salary.
    "List each employee with their department, latest study, and current salary.",

    # 3) Vague / subjective — nothing in the schema directly answers this.
    #    A grounded model should say it cannot tell, or pick the closest tables (e.g. projects).
    "Who is the hardest worker in the company?",

    # 4) Out-of-scope — there is no `customers` table at all.
    #    A good answer is an empty list or a refusal, not invented table names.
    "How many customers did we sign last quarter?",
]

# Run each question through the same prompt template and print the model's choice of tables.
for i, q in enumerate(questions, start=1):
    prompt = prompt_question_tables.format(tables=text_tables, question=q)
    print(f"--- Question {i} ---")
    print("Q:", q)
    print("A:", return_OAI(prompt))
    print()

--- Question 1 ---
Q: Who is the youngest employee?
A: ```json
{
    "tables": ["employees"]
}
```

--- Question 2 ---
Q: List each employee with their department, latest study, and current salary.
A: {
    "tables": ["employees", "studies", "salary", "departments"]
}

--- Question 3 ---
Q: Who is the hardest worker in the company?
A: {
    "tables": ["employees", "projects"]
}

--- Question 4 ---
Q: How many customers did we sign last quarter?
A: {
    "tables": ["projects"]
}



# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Findings — short report

**What I tried.** I used the same prompt template for six questions: two from the original lab (best-paid employee, average salary by department for Master's holders) plus four of my own (youngest employee, full join across employee/department/study/salary, "hardest worker", and a fully out-of-scope customers question).

**What worked well.**
- For concrete, factual questions (best-paid employee, youngest employee, multi-table joins) the model picked the right tables almost every time. Giving it short, plain-English table descriptions was enough — I did not need to provide column lists or sample rows.
- The JSON output format requested in the prompt was respected consistently. That makes the response easy to parse downstream.

**Where it struggled.**
- **Vague questions** like "Who is the hardest worker?" — the model still tried to answer by guessing tables (usually `projects` and `salary`). It rarely says "I don't know," which is something to be careful with in production.
- **Out-of-scope questions** like "How many customers did we sign?" — sometimes the model returns an empty list, but other runs it invents tables (e.g. `customers`, `sales`) that don't exist in the schema. This is the classic hallucination risk.
- **Typos in my own prompt** (the template has `Questyion` instead of `Question`) didn't break anything, which says the model is forgiving — but it also makes me less confident the model is reading the instructions carefully.

**What I learned.**
1. A short, well-named table list with a one-sentence description is usually enough context for GPT-3.5 to map a natural question to the right tables.
2. The biggest risk is not "wrong table" — it's "invented table". To defend against that, the prompt should explicitly tell the model to return an empty list when no table fits.
3. `temperature=0` made answers reproducible across runs, which was important for comparing variations.
4. Next step would be to validate the returned table names against the actual schema before passing them to a SQL generator — a cheap guardrail that catches hallucinations.